<a href="https://colab.research.google.com/github/Napatphorn/GE388-Lab2/blob/main/Lab2_Spatial_Analysis_6606614730.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. ติดตั้งไลบรารี geemap (รันแค่ครั้งแรกของ session)
!pip install geemap

# 2. Import ไลบรารีที่ต้องใช้
import ee
import geemap

# 3. ยืนยันตัวตนเข้าสู่ระบบ
ee.Authenticate()

# 4. เริ่มต้นการทำงาน
ee.Initialize(project='v6-dk-459909')

print("ตั้งค่า GEE สำเร็จแล้ว!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.4 MB/s eta 0:00:00
ตั้งค่า GEE สำเร็จแล้ว!


 นักศึกษาเลือก อำเภอแม่แจ่ม จังหวัดเชียงใหม่ สาเหตุที่เลือก: เป็นหนึ่งในพื้นที่ที่มีความเสี่ยงสูงด้านไฟป่าและปัญหาฝุ่นควัน การนำข้อมูลดาวเทียมมาวิเคราะห์เพื่อดูพืชพรรณที่หนาแน่นและความชื้นสะสม จะช่วยให้เห็นพื้นที่เสี่ยงที่อาจกลายเป็นเชื้อเพลิงไฟป่า โดยเฉพาะในช่วงฤดูแล้งที่มีการทำเกษตรกรรมบนที่สูง  การศึกษาพื้นที่นี้จะช่วยให้หน่วยงานเห็นภาพความแห้งแล้งที่ชัดเจน

In [16]:
import ee
import geemap

# สร้างแผนที่
Map = geemap.Map()

# ดึงข้อมูลขอบเขตระดับอำเภอจาก FAO GAUL (ระดับ 2 คือระดับอำเภอ)
districts = ee.FeatureCollection("FAO/GAUL/2015/level2")

# กรองเอาเฉพาะจังหวัดเชียงใหม่ (Chiang Mai) และ อำเภอแม่แจ่ม (Mae Chaem)
roi_mae_chaem = districts.filter(ee.Filter.And(
    ee.Filter.eq('ADM1_NAME', 'Chiang Mai'),
    ee.Filter.eq('ADM2_NAME', 'Mae Chaem')
))

# ปรับมุมมองแผนที่ให้ซูมไปที่อำเภอแม่แจ่ม
Map.centerObject(roi_mae_chaem, 10)

# แสดงขอบเขตอำเภอแม่แจ่มบนแผนที่
Map.addLayer(roi_mae_chaem, {}, 'Mae Chaem Boundary')

# แสดงแผนที่
Map

Map(center=[18.663102178810302, 98.31182557377413], controls=(WidgetControl(options=['position', 'transparent_…

เลือกใช้ดาวเทียม Sentinel-2: เนื่องจาก Sentinel-2 มีความละเอียดเชิงพื้นที่ที่ 10 เมตร ซึ่งเหมาะสมอย่างยิ่งกับภูมิประเทศของอำเภอแม่แจ่มที่เป็นภูเขาสลับซับซ้อน

เหตุผลในการเลือกช่วงเวลา:

เลือกช่วงเวลา 1 ม.ค. - 30 เม.ย. 2024 เนื่องจากเป็นช่วงกลางถึงปลายฤดูแล้งของภาคเหนือ ซึ่งเป็นช่วงที่พืชพรรณส่วนใหญ่ผลัดใบ แปลงเกษตรถูกปล่อยทิ้งร้าง นอกจากนี้ยังกำหนดเกณฑ์กรองเมฆไว้ที่ < 10% และใช้เทคนิค Median Composite เพื่อให้ได้ภาพที่เคลียร์ที่สุดสำหรับการวิเคราะห์ Index ในขั้นตอนต่อไป

In [17]:
# roi_mae_chaem = ee.FeatureCollection("FAO/GAUL/2015/level2").filter(ee.Filter.eq('ADM2_NAME', 'Mae Chaem'))

# 1. เลือกชุดข้อมูลดาวเทียม Sentinel-2 Surface Reflectance (SR)
s2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')

# 2. กรองข้อมูลตามพื้นที่ ช่วงเวลา และปริมาณเมฆ
# เลือกช่วง มกราคม - เมษายน ซึ่งเป็นจุดพีกของฤดูแล้งและปัญหาหมอกควัน/ไฟป่าในภาคเหนือ
s2_dry_season = (s2_collection
                 .filterBounds(roi_mae_chaem)
                 .filterDate('2024-01-01', '2024-04-30')
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))) # เลือกภาพที่มีเมฆน้อยกว่า 10%

# 3. สร้างภาพ Composite ตัวแทนของฤดูแล้ง
# ใช้การหาค่ามัธยฐาน (median) เพื่อกำจัด Noise หรือเมฆที่อาจหลงเหลืออยู่
# จากนั้นตัดภาพ (clip) ให้อยู่เฉพาะในพื้นที่ roi_mae_chaem
s2_composite = s2_dry_season.median().clip(roi_mae_chaem)

# 4. แสดงผลบนแผนที่ (ลองแสดงเป็น True Color ดูก่อน)
Map = geemap.Map()
Map.centerObject(roi_mae_chaem, 10)

# ตั้งค่าการแสดงผลแบบสีธรรมชาติ (Red=B4, Green=B3, Blue=B2)
vis_params_true = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000,
    'gamma': 1.4
}

Map.addLayer(s2_composite, vis_params_true, 'S2 True Color - Dry Season')
Map

Map(center=[18.66310217881025, 98.3118255737741], controls=(WidgetControl(options=['position', 'transparent_bg…

1.Spectral index พื้นที่ส่วนใหญ่ของจังหวัดเชียงใหม่ในช่วงฤดูแล้งมีค่า NDMI ต่ำ โดยเฉพาะบริเวณพื้นที่หุบเขาแม่น้ำและพื้นที่เกษตรกรรมที่เก็บเกี่ยวแล้ว สะท้อนถึงสภาวะที่เรือนยอดพืช แทบไม่มีความชื้นสะสมอยู่เลย ซึ่งสอดคล้องกับพฤติกรรมทางธรรมชาติของป่าเต็งรังและป่าเบญจพรรณในภาคเหนือที่จะผลัดใบในฤดูแล้ง  หน่วยงานจึงควรเฝ้าระวังพื้นที่สีแดงอย่างเข้มงวด ในทางกลับกัน พื้นที่ตามแนวเทือกเขาและยอดเขา ยังคงมีความชื้นสะสมอยู่ ข้อมูลนี้ช่วยให้หน่วยงานรัฐสามารถระบุพิกัดพื้นที่ป่าที่เสี่ยงต่อไฟป่า

In [12]:
# 1. คำนวณ NDMI (ใช้แบนด์ NIR และ SWIR1 ของ Sentinel-2)
# สมการ NDMI = (B8 - B11) / (B8 + B11)
ndmi = s2_composite.normalizedDifference(['B8', 'B11']).rename('NDMI')

# 2. ตั้งค่าสีสำหรับการแสดงผล (แดง=แห้งแล้ง, น้ำเงิน=ชื้น)
ndmi_vis = {
    'min': -0.2,
    'max': 0.4,
    'palette': ['d73027', 'fc8d59', 'fee090', 'e0f3f8', '91bfdb', '4575b4']
}

# 3. แสดงผลบนแผนที่
Map.addLayer(ndmi, ndmi_vis, 'NDMI - Dry Season')
Map

Map(bottom=468862.0, center=[18.786067249579002, 98.29879760742189], controls=(WidgetControl(options=['positio…

2. เปรียบเทียบ Composite ภาพเปรียบเทียบซ้าย-ขวาแสดงความแตกต่างอย่างเห็นได้ในช่วงเดือนพฤศจิกายน-ธันวาคม พื้นที่ส่วนใหญ่ของแม่แจ่มยังคงปกคลุมด้วยสีแดงสดถึงแดงเข้ม แต่เมื่อเลื่อนมาดูภาพช่วงเดือนมกราคม-เมษายน พื้นที่สีแดงสดเหล่านั้นหายไปกว่าครึ่ง และถูกแทนที่ด้วยโทนสีเทาอมน้ำตาล การลดลงของพื้นที่สีแดงสดหมายถึง สูญเสียความหนาแน่นของพืชพรรณและคลอโรฟิลล์ สีเทาอมน้ำตาลที่ปรากฏขึ้นมาแทนที่ สะท้อนถึงพื้นที่ดินเปล่า หรือซากพืชแห้ง หรืออาจรวมถึงพื้นที่ที่ถูกไฟไหม้และมีร่องรอยการเผา

In [18]:
# 1. สร้างภาพตัวแทนช่วงต้นฤดูแล้ง (เช่น พ.ย. - ธ.ค. 2023)
s2_early_dry = (s2_collection
                 .filterBounds(roi_mae_chaem)
                 .filterDate('2023-11-01', '2023-12-31')
                 .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
                 .median().clip(roi_mae_chaem))

# 2. ตั้งค่าแสดงผลแบบ False Color (เน้นพืชพรรณเป็นสีแดง)
false_color_vis = {'bands': ['B8', 'B4', 'B3'], 'min': 0, 'max': 3000}

# 3. สร้าง Tile Layer สำหรับใช้ทำ Split Map
left_layer = geemap.ee_tile_layer(s2_early_dry, false_color_vis, 'Nov/Dec (Early Dry)')
right_layer = geemap.ee_tile_layer(s2_composite, false_color_vis, 'Feb/Apr (Peak Dry)')

# 4. สร้างแผนที่ใหม่และเรียกใช้เครื่องมือ Split Map
Map = geemap.Map()
Map.centerObject(roi_mae_chaem, 10)

# จับเลเยอร์ยัดลงไปซ้าย-ขวา
Map.split_map(left_layer, right_layer)

Map

Map(center=[18.66310217881025, 98.3118255737741], controls=(ZoomControl(options=['position', 'zoom_in_text', '…

3. พื้นที่เข้าข่ายแห้งแล้งรุนแนง ตัวเลขเชิงปริมาณนี้ ช่วยวางแผนกำลังพลของหน่วยงานรัฐ การรู้ว่ามีพื้นที่วิกฤตแล้งจัดกี่ตารางกิโลเมตร ทำให้สามารถคำนวณได้ว่าต้องใช้เจ้าหน้าที่พิทักษ์ป่ากี่นายในการทำแนวกันไฟ หรือต้องเตรียมแผนเผชิญภัยแบบใด

In [11]:
# 1. สร้างเงื่อนไขพื้นที่แห้งแล้ง (NDMI น้อยกว่า -0.1)
severe_drought = ndmi.lt(-0.1)

# 2. คำนวณพื้นที่ (ตร.ม.) โดยเอา Pixel Area มาคูณกับพิกเซลที่เป็น True (1)
drought_area_img = severe_drought.multiply(ee.Image.pixelArea())

# 3. รวมพื้นที่ทั้งหมดในเขตแม่แจ่ม (Reduce Region)
drought_stats = drought_area_img.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=roi_mae_chaem.geometry(),
    scale=10, # ความละเอียด Sentinel-2
    maxPixels=1e10
)

# 4. แปลงหน่วยเป็น ตารางกิโลเมตร
area_sqkm = drought_stats.getNumber('NDMI').divide(1e6).getInfo()
print(f"พื้นที่ที่เข้าข่ายแห้งแล้งรุนแรง: {area_sqkm:.2f} ตารางกิโลเมตร")

# 5. Visualize พื้นที่แห้งแล้งรุนแรงบนแผนที่
Map.addLayer(severe_drought.selfMask(), {'palette': ['red']}, 'Severe Drought Areas')

พื้นที่ที่เข้าข่ายแห้งแล้งรุนแรง: 766.29 ตารางกิโลเมตร


In [15]:
# ก่อนรันโค้ดนี้ ตรวจสอบให้แน่ใจว่าได้เชื่อมต่อ Google Drive กับ Colab แล้ว
# โค้ดสำหรับ Export ภาพ NDMI ไปยัง Google Drive
geemap.ee_export_image_to_drive(
    image=ndmi, # ตัวแปรภาพที่เราต้องการ Export (เปลี่ยนเป็นชื่อตัวแปรของคุณถ้าใช้ชื่ออื่น)
    description='Mae_Chaem_NDMI_Dry_Season_2024', # ชื่อ Task ที่จะปรากฏในแท็บ Tasks ของ GEE
    folder='GEE_Lab2_Exports', # ชื่อโฟลเดอร์ที่จะถูกสร้างขึ้นมาใหม่ใน Google Drive ของคุณ
    region=roi_mae_chaem.geometry(), # ขอบเขตพื้นที่ที่จะ Export (แม่แจ่ม)
    scale=10, # ความละเอียด (Spatial Resolution) ในหน่วยเมตร
    crs='EPSG:32647', # ระบบพิกัด (Coordinate Reference System)
    maxPixels=1e13, # ป้องกัน Error กรณีภาพใหญ่เกินไป
    fileFormat='GeoTIFF' # นามสกุลไฟล์
)

print("กำลังส่งออกข้อมูลไปยัง Google Drive... โปรดตรวจสอบความคืบหน้าในแถบ Tasks ของ Earth Engine หรือใน Google Drive ของคุณ")

กำลังส่งออกข้อมูลไปยัง Google Drive... โปรดตรวจสอบความคืบหน้าในแถบ Tasks ของ Earth Engine หรือใน Google Drive ของคุณ
